# Plank form classifier — Colab training runner

Trains the OPTIONAL plank assist model (`hips_low / correct / hips_high`) as a small Keras MLP and
exports `plank_form.tflite`. Plank stays **rule-based**; this model is only an assist hint.

Unlike the push-up runner, this needs **no Kaggle and no MediaPipe**: the Vollkorn01 dataset is read
over HTTP by `plank_pose_dataset.load_plank_dataframe()`, and the 5 features are the
rotation/scale-normalized **body-frame** features (identical to the app's `:core` PlankFeatureExtractor).

Run top to bottom. Mount Drive with the **kimgt2828** account.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/health_training'
RUNS_DIR   = f'{DRIVE_ROOT}/runs'
import os; os.makedirs(RUNS_DIR, exist_ok=True)
print(RUNS_DIR)

In [ ]:
# Clone the model-training branch (has the plank code). %cd /content first = re-run safe.
%cd /content
!rm -rf /content/health_trainer
!git clone --branch model-training --single-branch https://github.com/kimgt0128/health-trainer.git /content/health_trainer
%cd /content/health_trainer
# tensorflow / scikit-learn / pandas are preinstalled on Colab; install only to be safe.
!pip install -q "scikit-learn>=1.4" "pandas>=2.0"

In [ ]:
# === TRAIN === reads the Vollkorn01 dataframe over HTTP, builds body-frame features, trains a Keras
# MLP (class weights + stratified 5-fold CV), writes 4 artifacts. Takes ~1-2 minutes.
%cd /content/health_trainer
import os
os.environ['PYTHONPATH'] = '/content/health_trainer/ml/src'
!python ml/src/train_plank_form_mlp.py \
    --run-dir "$RUNS_DIR/plank_form_classifier_v1" \
    --test-size 0.2 \
    --random-state 42 \
    --hidden 16 \
    --epochs 100

In [ ]:
# Inspect the artifacts written to Drive (4 files expected) + the metrics.
!find "$RUNS_DIR/plank_form_classifier_v1" -maxdepth 1 -type f -print
!echo '--- metrics_summary.json ---'
!cat "$RUNS_DIR/plank_form_classifier_v1/metrics_summary.json"

## After training

If `held_out` / `cv` look good (macro-F1 ≥ 0.75, no class F1 < 0.60 — watch the rare `hips_low`),
copy `plank_form.tflite` into the app to enable the assist:

```
app/src/main/assets/models/plank_form.tflite
```

The app already wires it (`FormClassifierRegistry` PLANK entry, 750 ms hold-throttled). Until the
file is present, the app stays rules-only and never crashes.